<a href="https://colab.research.google.com/github/yadavlakshya743/Neural-Machine-Translation-System/blob/main/experiments/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Machine Translation: English <-> Hindi (Transformer Seq2Seq)
**Grounding:** *Attention Is All You Need* (Vaswani et al., NeurIPS 2017)  
**Dataset:** Official IIT Bombay English-Hindi Parallel Corpus  
**Target Benchmark:** SacreBLEU >= 20.0, chrF++ >= 45.0 on official 2,507 test split.

---
### 1. Verify GPU Hardware Acceleration
Ensure you have selected **Runtime > Change runtime type > T4 GPU**.

In [2]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Fri Oct 16 10:05:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Clone GitHub Repository & Install Dependencies

In [3]:
!rm -rf Neural-Machine-Translation-System
!git clone https://github.com/yadavlakshya743/Neural-Machine-Translation-System.git
%cd Neural-Machine-Translation-System
!pip install -r requirements.txt

Cloning into 'Neural-Machine-Translation-System'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 95 (delta 17), reused 94 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 95.94 KiB | 6.00 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/Neural-Machine-Translation-System
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### 3. Acquire Corpus & Train Tokenizers (Step 1)
Downloads the official IIT Bombay splits and trains English & Hindi SentencePiece BPE tokenizers.

In [4]:
!python scripts/prepare_data.py --max_train_samples 46857 --vocab_size 8000

[*] Downloading https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/main/data/validation-00000-of-00001.parquet -> data/raw/validation.parquet...
validation.parquet: 100% 85.7k/85.7k [00:00<00:00, 386kB/s]
[+] Successfully downloaded: data/raw/validation.parquet
[*] Downloading https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/main/data/test-00000-of-00001.parquet -> data/raw/test.parquet...
test.parquet: 100% 500k/500k [00:00<00:00, 1.60MB/s]
[+] Successfully downloaded: data/raw/test.parquet
[*] Downloading https://huggingface.co/datasets/cfilt/iitb-english-hindi/resolve/main/data/train-00000-of-00001.parquet -> data/raw/train.parquet...
train.parquet: 100% 190M/190M [00:02<00:00, 81.2MB/s]
[+] Successfully downloaded: data/raw/train.parquet
[*] Reading and processing data/raw/validation.parquet...
[+] Retained 520 pairs from 520 rows (is_eval_split=True).
[*] Reading and processing data/raw/test.parquet...
[+] Retained 2,507 pairs from 2,507 rows (is_eval

### 4. Train Custom Transformer Seq2Seq Model on GPU (Step 2)
Trains 4 Encoder layers + 4 Decoder layers with Multi-Head Attention on 46k parallel pairs for 15 epochs.

In [5]:
!python scripts/train.py --epochs 15 --batch_size 64 --lr 0.0005

      TRAINING CUSTOM TRANSFORMER NMT (Attention Is All You Need)     
[*] Configuration:   ml/configs/baseline_config.yaml
[*] Hardware Device: CUDA
[*] Epochs:          15
[*] Batch Size:      64
[*] Learning Rate:   0.0005
[*] Target Checkpt:  models/best_transformer_en_hi.pt
[+] Loaded Tokenizers | EN Vocab: 8,000 | HI Vocab: 8,000
[*] Loading training corpus...
[+] Loaded 5,000 training pairs.
[*] Loading validation benchmark (official dev split)...
[+] Loaded 520 validation pairs.
[+] Transformer Model Instantiated | Trainable Parameters: 13,525,824
[*] Commencing Training Loop for 15 Epoch(s)...
Epoch 01 [Train]: 100% 79/79 [00:05<00:00, 15.01batch/s, loss=5.5748, avg_loss=6.2809]
           [Val]  : 100% 9/9 [00:00<00:00, 18.16batch/s, val_loss=9.4679, avg_val=9.4670]
[*] Epoch 01/15 Complete | Train Loss: 6.2809 | Val Loss: 9.4670 | Time: 5.8s
[+] Best checkpoint saved: models/best_transformer_en_hi.pt (Val Loss: 9.4670)
Epoch 02 [Train]: 100% 79/79 [00:04<00:00, 19.25batch/s,

### 5. Evaluate on Official 2,507 Test Benchmark (SacreBLEU & chrF++)
Runs beam search decoding on all 2,507 sentences of the isolated test set and computes final metrics.

In [ ]:
!python scripts/evaluate.py --num_samples 0 --strategy beam_search --beam_size 4 --output_json experiments/colab_final_experiment.json

      OFFICIAL BENCHMARK EVALUATION (SacreBLEU & chrF++)      
[*] Checkpoint:     models/best_transformer_en_hi.pt
[*] Device:         CUDA
[*] Test Source:    data/processed/test_en.txt
[*] Test Reference: data/processed/test_hi.txt
[*] Strategy:       beam_search
[+] Loaded Model Checkpoint (Epoch: 3, Train Loss: 5.018179458907888)
[+] Evaluating on 2,507 of 2,507 official benchmark sentences.
[*] Translating test set...
Evaluating:  67% 1689/2507 [1:26:35<39:36,  2.91s/sent]

### 6. Download Checkpoint & Evaluation Report to Local Machine
Download the trained  model and the evaluation report directly to your computer.

In [ ]:
from google.colab import files
import os

if os.path.exists("models/best_transformer_en_hi.pt"):
    files.download("models/best_transformer_en_hi.pt")
    print("[+] Downloaded trained model checkpoint.")

if os.path.exists("experiments/colab_final_experiment.json"):
    files.download("experiments/colab_final_experiment.json")
    print("[+] Downloaded evaluation report JSON.")